In [1]:
import numpy as np
import pandas as pd

In [2]:
jobs_clean_df = pd.read_csv("jobs_clean.csv")
jobs_clean_df.head()

,Unnamed: 0,id,title,contract_type,description,created,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,year_week,clean_city,clean_company
0,0,5753281217,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,R,Onsite,Intern,2026-06-01/2026-06-07,Österreich,STRABAG BRVZ GMBH
1,1,5753281216,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,R,Onsite,Intern,2026-06-01/2026-06-07,Spittal an der Drau,STRABAG BRVZ GMBH
2,2,5762094132,Data Analyst - MS PowerBI (m/w/d),NaN,Baustoff + Metall ist ein hoch spezialisierter...,2026-06-13 06:58:14,AT,Not Found,"SQL, R",Hybrid,Intern,2026-06-08/2026-06-14,"Wien, Österreich",Baustoff + Metall Gesellschaft m.b.H. Österrei...
3,3,5727781690,Data Analyst (m/w/d),permanent,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,2026-05-13 08:03:25,AT,Not Found,"SQL, R",Hybrid,Intern,2026-05-11/2026-05-17,"Josefstadt, Wien",ÖSW Österreichisches Siedlungswerk Gemeinnützi...
4,4,5758608143,Data Analyst,NaN,"What are you working on?\nGenres: Casual, Puzz...",2026-06-10 17:18:19,AT,Not Found,"PYTHON, SQL, TABLEAU, R",Unspecified,Unspecified,2026-06-08/2026-06-14,Österreich,Hitapps


In [ ]:
# Get the top 10 employers by counting occurrences in the 'clean_company' column
top_employers = jobs_clean_df["clean_company"].value_counts().head(10)
print("=== Top 10 Employers ===")
print(top_employers)

# Extract the names of the top 5 employers
top_5_companies = jobs_clean_df["clean_company"].value_counts().head(5).index

# Filter the original DataFrame to keep only rows belonging to these top 5 companies
filtered_df = jobs_clean_df[jobs_clean_df["clean_company"].isin(top_5_companies)]

# Create a pivot table to see the distribution of work modes (e.g., Remote, Hybrid) for the top 5 employers
pivot_employer_work_mode = filtered_df.pivot_table(
    index="clean_company",
    columns="work_mode",
    values="id",
    aggfunc="count",
    fill_value=0,
)


print("\n=== Work Mode Distribution of Top Employers ===")
print(employer_work_mode_matrix)

=== Top 10 Employers ===
clean_company
Externatic                                235
Unknown                                   154
Smart Future Campus GmbH                  105
Collective.work                           105
ITOL Recruit                               88
Securitas                                  68
ISCOD                                      49
Harnham - Data & Analytics Recruitment     43
Link Group                                 43
Citigroup                                  34
Name: count, dtype: int64

=== Work Mode Distribution of Top Employers ===
work_mode                 Hybrid  Onsite  Remote  Unspecified
clean_company                                                
Collective.work               16       1       0           88
Externatic                    59       5      26          145
ITOL Recruit                   2       0       0           86
Smart Future Campus GmbH       0       0     105            0
Unknown                       35      18      36      

In [ ]:
# Split the comma-separated skills into lists and explode them so each skill gets its own row
exploded_df = jobs_clean_df.assign(extracted_skills=jobs_clean_df["extracted_skills"].str.split(", ")).explode("extracted_skills")

#Filter out rows where no skills were specified ("None Mentioned")
valid_skills_df = exploded_df[exploded_df["extracted_skills"] != "None Mentioned"]

# Creat pivot table of skills vs. seniority levels
# normalized by row ('index') and multiplied by 100 to get percentages
pivot_skill_seniority = pd.crosstab(
    valid_skills_df["extracted_skills"], 
    valid_skills_df["seniority_level"], 
    normalize='index'  
) * 100

print("=== Skill Distribution by Seniority Level (Percentage) ===")
print(skill_seniority.round(2).sort_values(by="Senior", ascending=False))

=== Skill Distribution by Seniority Level (Percentage) ===
seniority_level   Intern  Junior  Middle  Senior  Unspecified
extracted_skills                                             
AWS                10.17    1.72    0.13   72.26        15.72
EXCEL              13.85    3.66    0.10   67.20        15.19
SPARK              11.43    2.20    0.22   64.18        21.98
SAS                16.95    1.69    0.00   60.68        20.68
PYTHON             13.49    3.52    0.18   60.67        22.15
SQL                17.00    3.15    0.09   57.47        22.29
TABLEAU            16.75    2.67    0.07   56.02        24.49
POWER BI           19.31    3.44    0.05   54.73        22.47
R                  16.47    2.77    0.07   54.26        26.43
AZURE              21.28    2.36    0.23   52.59        23.54


In [ ]:
# Filter out rows where work_mode is 'Unspecified' or 'Unknown'
valid_df = valid_skills_df[~valid_skills_df["work_mode"].isin(["Unspecified", "Unknown"])]

# Identify the top 10 most frequent skills based on value counts
top_10_skills = valid_df["extracted_skills"].value_counts().head(10).index
# Filter the dataframe to retain only rows containing these top 10 skills
filtered_skills_df = valid_df[valid_df["extracted_skills"].isin(top_10_skills)]

# find the percentage distribution of work modes for each skill
skill_work_mode = pd.crosstab(
    filtered_skills_df["extracted_skills"], 
    filtered_skills_df["work_mode"], 
    normalize="index"# Normalize along rows to get percentages per skill
) * 100

print("=== Work Mode Distribution for Top 10 Skills (%) ===")
# Print the rounded distribution data, sorted by the percentage of 'Remote' work in descending order
print(skill_work_mode.round(1).sort_values(by="Remote", ascending=False))

=== Work Mode Distribution for Top 10 Skills (%) ===
work_mode         Hybrid  Onsite  Remote
extracted_skills                        
AZURE               51.2     7.2    41.6
SQL                 49.7    16.3    34.0
AWS                 57.8     9.1    33.1
POWER BI            51.4    15.6    33.0
PYTHON              52.9    16.3    30.9
SPARK               60.3     9.4    30.3
R                   51.7    19.6    28.7
TABLEAU             51.9    21.4    26.7
EXCEL               54.3    22.8    22.8
SAS                 58.5    22.0    19.5
